# 17 - Collect a CartPole Dataset

This notebook collects CartPole the same way `01_collect_dataset.ipynb` collects FrozenLake:

1. Build environments with `mouse_gym.EnvConfig` and `make_group_env`.
2. Create one `Datastore` for each environment stream.
3. Step the grouped environment and append each flat row.
4. Publish the stores with `push_stores_to_hub`.

CartPole's observation is a length-4 float vector, not a discrete id. The row stores that vector as `observation`. `18_train_offline_cartpole_dqn.ipynb` reads each element with a text field `input_index`.

The four elements, in Gymnasium order, are cart position, cart velocity, pole angle (radians), and pole angular velocity. Actions are discrete: `0` or `1`. There is no optimal-Q field, so this collector takes a random action every step.

`MAX_EPISODE_STEPS` is `50` and `MAX_TASK_EPISODES` is `4`, shorter than CartPole-v1's usual 500-step episode, so one task stays inside the training notebook's token budget. A step that keeps the pole up scores `1`. The episode ends when the pole falls, the cart leaves the track, or the step cap is hit.

This is a short usage example, not a full experiment. Scale collection up in [mouse-experiment](https://github.com/micahr234/mouse-experiment).


In [ ]:
import numpy as np

from mouse_gym import EnvConfig, make_group_env
from mouse_core.data import Datastore, push_stores_to_hub


DATASET_ID = "mouse-example-cartpole-dataset"  # Hugging Face dataset repo for push_stores_to_hub
NUM_ENVS = 4                                    # number of environment streams in the GroupEnv
STEPS_PER_ENV = 256                             # rows collected for each datastore
MAX_EPISODE_STEPS = 50                          # max steps per episode (shorter than CartPole-v1's 500)
MAX_TASK_EPISODES = 4                           # max episodes per task


## Build Environment

`id="CartPole-v1"` is the Gymnasium env. `make_group_env` steps every stream with one call. `seed` is drawn once per task and passed to `reset(seed=...)` only at task start.

`max_task_episodes` is the task-length timeout (`task_done=2`). `kwargs["max_episode_steps"]` is the episode cap (`episode_done=2` when it fires; a fall or a cart leaving the track is `episode_done=1`).


In [ ]:
configs = [
    EnvConfig(
        id="CartPole-v1",
        name=f"cartpole_{i}",
        seed=i,
        max_task_episodes=MAX_TASK_EPISODES,
        kwargs={"max_episode_steps": MAX_EPISODE_STEPS},
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)


## Create Datastores

One `Datastore` per environment stream. Each store keeps that stream's rows in order. `DataLoader` mixes streams later.


In [ ]:
stores = [Datastore(name=name) for name in env.names]  # one ordered store per env stream


## Collect Transitions

Each step merges the action you sent with the environment output. `observation` stays a length-4 vector. CartPole's `info` is empty, so it is dropped and the row stays flat.

The first `env.step` also resets each stream. That row is stored, so every stream starts at its first observed state. Reward on that reset frame is `0`. Later steps score `1` while the pole is up.


In [ ]:
def run_rollout(*, env, stores: list, steps_per_env: int) -> None:
    """Step environments at random and append flat rows into per-stream datastores."""
    env.metrics.clear()
    for step in range(steps_per_env):
        inputs = env.sample_random_input()
        outputs = env.step(inputs)
        for store, inp, out in zip(stores, inputs, outputs):
            row = {**inp, **out}
            row.pop("info")
            store.append(data=row)
        if (step + 1) % 64 == 0 or (step + 1) == steps_per_env:
            task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
            mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
            print(
                f"step {step + 1}/{steps_per_env} | "
                f"{len(task_scores)} tasks | mean task return {mean_task_score:.1f}"
            )
            env.metrics.clear()

run_rollout(env=env, stores=stores, steps_per_env=STEPS_PER_ENV)
env.close()


## Push to the Hub

`push_stores_to_hub` uploads every datastore to one Hugging Face dataset repository. Each store is one config. `18_train_offline_cartpole_dqn.ipynb` reloads them with `load_stores_from_hub`.


In [ ]:
url = push_stores_to_hub(stores=stores, repo_id=DATASET_ID, split="train", private=False)
print(f"Pushed to {url}")
